In [1]:
import numpy as np

SKI_CONF_MIN = 0.15  # pseudo ski keypoints below this are treated as missing


def compute_skier_orientation_features(keypoints_21):
    """
    Computes body angle, ski orientations, and grab distances from a [21, 3] frame keypoint array.
    keypoints_21: [21, (x, y, conf)]
    Returns: dict of derived physical metrics
    """
    kpts = keypoints_21[:, :2]
    confs = keypoints_21[:, 2] if keypoints_21.shape[1] > 2 else np.ones(len(keypoints_21))

    # --- 1. Body Orientation Angle (Spine Vector) ---
    l_shoulder, r_shoulder = kpts[5], kpts[6]
    l_hip, r_hip = kpts[11], kpts[12]

    shoulder_center = (l_shoulder + r_shoulder) / 2.0
    hip_center = (l_hip + r_hip) / 2.0
    spine_vec = shoulder_center - hip_center  # Vector pointing from hips to shoulders

    # Angle relative to vertical axis (in degrees, -180 to 180)
    body_angle = np.degrees(np.arctan2(spine_vec[0], -spine_vec[1]))

    # --- 2. Ski Angles & Relative Crossing Angle ---
    l_ski_tip, l_ski_tail = kpts[17], kpts[18]
    r_ski_tip, r_ski_tail = kpts[19], kpts[20]
    left_ok = confs[17] >= SKI_CONF_MIN and confs[18] >= SKI_CONF_MIN
    right_ok = confs[19] >= SKI_CONF_MIN and confs[20] >= SKI_CONF_MIN

    l_ski_vec = l_ski_tip - l_ski_tail
    r_ski_vec = r_ski_tip - r_ski_tail

    angle_l_ski = np.degrees(np.arctan2(l_ski_vec[1], l_ski_vec[0])) if left_ok else np.nan
    angle_r_ski = np.degrees(np.arctan2(r_ski_vec[1], r_ski_vec[0])) if right_ok else np.nan

    # Absolute crossing angle between the two skis
    if left_ok and right_ok:
        dot_product = np.dot(l_ski_vec, r_ski_vec)
        norms = np.linalg.norm(l_ski_vec) * np.linalg.norm(r_ski_vec)
        if norms > 1e-6:
            cos_angle = np.clip(dot_product / norms, -1.0, 1.0)
            crossed_ski_angle = np.degrees(np.arccos(cos_angle))
        else:
            crossed_ski_angle = 0.0
    else:
        crossed_ski_angle = np.nan

    # --- 3. Grab Distances (Wrist to Ski Tip/Tail) ---
    l_wrist, r_wrist = kpts[9], kpts[10]
    left_hand_ski_dist = min(
        np.linalg.norm(l_wrist - l_ski_tip) if left_ok else np.inf,
        np.linalg.norm(l_wrist - r_ski_tip) if right_ok else np.inf
    )
    right_hand_ski_dist = min(
        np.linalg.norm(r_wrist - l_ski_tip) if left_ok else np.inf,
        np.linalg.norm(r_wrist - r_ski_tip) if right_ok else np.inf
    )
    if not np.isfinite(min(left_hand_ski_dist, right_hand_ski_dist)):
        min_grab = np.nan
    else:
        min_grab = float(min(left_hand_ski_dist, right_hand_ski_dist))

    return {
        "body_angle": body_angle,
        "l_ski_angle": angle_l_ski,
        "r_ski_angle": angle_r_ski,
        "crossed_ski_angle": crossed_ski_angle,
        "min_grab_dist": min_grab
    }

## Ski detection (classical CV, no labels)

The pose model only predicts COCO body joints, so skis are found separately inside each tracked skier ROI.
The implementation lives in `ski_detector.py` (next to this notebook):

1. crop a square region centred on the ankles (bbox fallback when pose is missing),
2. enhance contrast and run LSD line-segment detection,
3. merge collinear fragments and keep the best nearly-parallel pair near the boots,
4. smooth angle / centre / length over time and carry the pair through short occlusions,
5. when the pair is lost, propagate the previous skis with Lucas–Kanade optical flow.

The **pose crop** is rotated with the smoothed hips→shoulders axis so the skier stays upright
(`ALIGN_POSE_ROI = True`); this roughly doubled pose coverage on the HD clips. The ski ROI is
left axis-aligned (the rotated variant was tested and was not better), and optical flow carries
the skis through missed frames.

Outputs per clip: `ski_dataset/<stem>_ski.npz` (2 ski quads per frame + confidence/angle/length)
and `roi_pose_dataset/<stem>_pose21.npy` (17 body + 4 pseudo ski keypoints = 21, ready for
`compute_skier_orientation_features`).


In [2]:
# --- Ski detector setup (imports the implementation in ski_detector.py) ---
from pathlib import Path

# Reload ski_detector so edits to the module are picked up without restarting
# the kernel (a stale import is the usual cause of "cannot import name ...").
import importlib
import ski_detector
importlib.reload(ski_detector)

# Same for temporal_tracker: a kernel that imported the old module would
# otherwise raise "unexpected keyword argument 'track_mode'"
import temporal_tracker
importlib.reload(temporal_tracker)

from ski_detector import (SkiDetectorCV, BodyAligner, skis_to_keypoints,
                          attach_ski_keypoints, draw_skis,
                          pose_keypoints_in_roi, make_roi_transform,
                          pose_keypoints_detected_roi, bbox_from_keypoints)

SKI_OUTPUT_DIR = Path("ski_dataset_xg2025men")
SKI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("SkiDetectorCV ready; ski tracks will be written to", SKI_OUTPUT_DIR.resolve())


In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# ==============================================================================
# 0. CONFIGURATION & INPUT INTEGRITY CHECKS
# ==============================================================================

DATA_DIR = Path("/Users/msserralheiro/Desktop/M_DS/CV/CV_Project/xg2025men")
OUTPUT_DIR = Path("roi_pose_dataset_xg2025men")
SKI_OUTPUT_DIR = Path("ski_dataset_xg2025men")   # one .npz of ski tracks per video
DETECT_SKIS = False                    # B1: ski detection is disabled (unreliable)
USE_DETECTED_ROI_POSE = True           # detect person per frame, then refine pose on their ROI
POSE_CONF = 0.1                        # low threshold needed for 320x240 AQA clips
BODY_BOX_PAD = 0.25                    # pad the pose-derived body box by 25%
USE_TEMPORAL_TRACKING = True           # offline Viterbi stitching + gap filling
TEMPORAL_MAX_GAP = 15                  # max frames to bridge between detections
OVERWRITE_POSE = False                 # default for any single call
OVERWRITE_POSE_AQA = True              # re-track with the whole-clip tracker
OVERWRITE_POSE_COMPETITION = True      # re-track with the whole-clip tracker
OVERWRITE_SKI = False                  # True re-processes clips that already have skis
ALIGN_POSE_ROI = True                  # rotate the pose crop with the skier's body axis
POSE_IMGSZ_AQA = 640                   # AQA clips are 320x240 / 640x480
POSE_IMGSZ_COMPETITION = 1280          # broadcast clips are 1280x720 wide shots
TRACK_MODE_AQA = "longest"             # AQA clips can contain spectators too
TRACK_MODE_COMPETITION = "longest"     # all people, keep the one present longest
CUT_AWARE = True                       # camera cuts: track every shot, keep all
PERSON_SELECT_AQA = "event"            # jump/rotation segment beats a longer approach
PERSON_SELECT_COMPETITION = "motion"   # jump motion, 80% stability floor
SKI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- dataset sources ----------------------------------------------------------
# AQA-7 clips (labels_CV.csv) and the new competition clips in competition_data/.
# Set INCLUDE_* to choose what this run processes.  MAX_CLIPS_PER_SOURCE is a
# per-source smoke-test limit (0 = process everything).
INCLUDE_AQA = False
INCLUDE_COMPETITION = True
MAX_CLIPS_PER_SOURCE = 0

labels = pd.read_csv(Path("/Users/msserralheiro/Desktop/M_DS/CV/CV_Project/labels_xg2025men.csv"), sep=",")
labels["video_path"] = labels["filename"].apply(
    lambda s: DATA_DIR / f"{s}.mp4"
).apply(lambda p: p if p.is_file() else None)
labels["dataset"] = "competition"
print(f"labels_combined rows: {len(labels)}")
print(labels.groupby("dataset").agg(
    clips=("filename", "size"),
    scored=("target", "count"),
    falls=("fall", "sum"),
).to_string())
print("All input integrity checks passed successfully.\n")

# ==============================================================================
# 1. DYNAMIC TRACKER & MOTION COMPENSATION MODULES
# ==============================================================================

class GlobalMotionCompensator:
    """Estimates camera movement (pan, tilt, zoom) via ORB optical flow."""
    def __init__(self):
        self.prev_gray = None
        self.detector = cv2.ORB_create(nfeatures=500)

    def compensate(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.prev_gray is None:
            self.prev_gray = gray
            return np.eye(2, 3, dtype=np.float32)

        kp1, des1 = self.detector.detectAndCompute(self.prev_gray, None)
        kp2, des2 = self.detector.detectAndCompute(gray, None)

        if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
            self.prev_gray = gray
            return np.eye(2, 3, dtype=np.float32)

        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = matcher.match(des1, des2)
        matches = sorted(matches, key=lambda x: x.distance)[:50]

        pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

        M, _ = cv2.estimateAffinePartial2D(pts1, pts2)
        self.prev_gray = gray
        return M if M is not None else np.eye(2, 3, dtype=np.float32)


class DynamicPersonTracker:
    """Tracks target skier across camera motion and performs automatic recovery."""
    def __init__(self, pose_model_path="yolov8n-pose.pt"):
        self.model = YOLO(pose_model_path)
        self.gmc = GlobalMotionCompensator()
        self.tracker = None
        self.bbox = None
        self.is_tracking = False

    def _create_csrt_tracker(self):
        """Robust instantiator for CSRT tracker across different OpenCV versions."""
        if hasattr(cv2, 'TrackerCSRT_create'):
            return cv2.TrackerCSRT_create()
        elif hasattr(cv2, 'legacy') and hasattr(cv2.legacy, 'TrackerCSRT_create'):
            return cv2.legacy.TrackerCSRT_create()
        elif hasattr(cv2, 'TrackerCSRT') and hasattr(cv2.TrackerCSRT, 'create'):
            return cv2.TrackerCSRT.create()
        elif hasattr(cv2, 'TrackerMIL_create'):
            print("  [WARN] CSRT unavailable (OpenCV 5); using MIL tracker instead.")
            return cv2.TrackerMIL_create()
        else:
            raise AttributeError(
                "CSRT Tracker is unavailable. Make sure 'opencv-contrib-python' "
                "is installed correctly: pip install opencv-contrib-python"
            )

    def init_tracker(self, frame, bbox):
        self.bbox = bbox
        self.tracker = self._create_csrt_tracker()
        self.tracker.init(frame, tuple(map(int, bbox)))
        self.is_tracking = True

    def update(self, frame):
        if not self.is_tracking:
            return None, False

        # Apply camera motion shift
        M = self.gmc.compensate(frame)
        if M is not None:
            cx = self.bbox[0] + self.bbox[2] / 2.0
            cy = self.bbox[1] + self.bbox[3] / 2.0
            pts = np.array([[[cx, cy]]], dtype=np.float32)
            transformed = cv2.transform(pts, M)
            new_cx, new_cy = transformed[0][0]
            self.bbox = (new_cx - self.bbox[2] / 2.0, new_cy - self.bbox[3] / 2.0, self.bbox[2], self.bbox[3])

        success, tracked_box = self.tracker.update(frame)
        if success:
            self.bbox = tracked_box
            return self.bbox, True
        else:
            self.is_tracking = False
            return self.bbox, False

    def recover_target(self, frame, prev_bbox, iou_threshold=0.2):
        """Full-frame re-detection fallback when visual tracking drifts."""
        results = self.model(frame, verbose=False, conf=0.1)[0]
        if results.boxes is None or len(results.boxes) == 0:
            return False

        best_iou, best_box = 0.0, None
        px, py, pw, ph = prev_bbox
        p_area = pw * ph

        for box in results.boxes.xywh.cpu().numpy():
            cx, cy, w, h = box
            bx, by = cx - w / 2.0, cy - h / 2.0

            inter_x1, inter_y1 = max(px, bx), max(py, by)
            inter_x2, inter_y2 = min(px + pw, bx + w), min(py + ph, by + h)
            inter_w, inter_h = max(0.0, inter_x2 - inter_x1), max(0.0, inter_y2 - inter_y1)
            inter_area = inter_w * inter_h
            union_area = p_area + (w * h) - inter_area
            iou = inter_area / union_area if union_area > 0 else 0

            if iou > best_iou and iou > iou_threshold:
                best_iou = iou
                best_box = (bx, by, w, h)

        if best_box is not None:
            self.init_tracker(frame, best_box)
            return True
        return False


def remap_crop_keypoints(kpts, scale, pad_x, pad_y, crop_x1, crop_y1):
    """Maps keypoint predictions from 640x640 cropped input back to video space."""
    if kpts is None or len(kpts) == 0:
        return np.array([])
    remapped = kpts.copy()
    remapped[..., 0] = (remapped[..., 0] - pad_x) / scale + crop_x1
    remapped[..., 1] = (remapped[..., 1] - pad_y) / scale + crop_y1
    return remapped


def find_skier_entry_frame(video_path, model, min_conf=0.15, max_scan_frames=150):
    """Scans video until the skier enters the frame."""
    cap = cv2.VideoCapture(str(video_path))
    frame_idx = 0

    while cap.isOpened() and frame_idx < max_scan_frames:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False, conf=min_conf)[0]
        if results.boxes is not None and len(results.boxes) > 0:
            persons = [b for b in results.boxes if int(b.cls[0]) == 0 and float(b.conf[0]) >= min_conf]
            if persons:
                best_person = max(persons, key=lambda b: float(b.conf[0]))
                cx, cy, w, h = best_person.xywh.cpu().numpy()[0]
                init_box = (cx - w / 2.0, cy - h / 2.0, w, h)
                cap.release()
                return frame_idx, frame, init_box
        frame_idx += 1

    cap.release()
    return None, None, None


# ==============================================================================
# 2. MAIN DATASET PROCESSING PIPELINE
# ==============================================================================

def process_aqa_dataset(labels_df, output_dir=OUTPUT_DIR, imgsz=640,
                        track_mode="longest", overwrite=OVERWRITE_POSE,
                        cut_aware=True, person_select="longest"):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    if USE_TEMPORAL_TRACKING:
        import importlib
        import temporal_tracker
        importlib.reload(temporal_tracker)
        from temporal_tracker import process_aqa_dataset_temporal
        return process_aqa_dataset_temporal(
            labels_df, output_dir=output_path, ski_output_dir=SKI_OUTPUT_DIR,
            data_dir=DATA_DIR, overwrite=overwrite,
            max_gap=TEMPORAL_MAX_GAP, pose_conf=POSE_CONF, bbox_pad=BODY_BOX_PAD,
            imgsz=imgsz, track_mode=track_mode, cut_aware=cut_aware,
            person_select=person_select)
    
    pipeline = DynamicPersonTracker(pose_model_path="yolov8n-pose.pt")
    pipeline.model.overrides["imgsz"] = int(imgsz)
    ski_detector = SkiDetectorCV(align_body=False, flow_propagation=True) if DETECT_SKIS else None
    pose_aligner = BodyAligner() if ALIGN_POSE_ROI else None
    processed_records = []

    for idx, row in labels_df.iterrows():
        video_file = row["video_path"]
        target_score = row["target"]
        stem_name = Path(video_file).stem

        print(f"[{idx+1}/{len(labels_df)}] Processing: {stem_name} (Target Score: {target_score})")

        ski_file = SKI_OUTPUT_DIR / f"{stem_name}_ski.npz"
        if ski_detector is not None and ski_file.exists() and not OVERWRITE_SKI:
            pose_file = output_path / f"{stem_name}_pose.npy"
            n_existing = int(np.load(pose_file).shape[0]) if pose_file.exists() else 0
            print(f"  [SKIP] ski track already exists ({n_existing} frames)")
            processed_records.append({
                "filename": row["filename"],
                "pose_feature_path": str(pose_file),
                "ski_feature_path": str(ski_file),
                "num_frames": n_existing,
                "target": target_score,
            })
            continue
        if ski_detector is not None:
            ski_detector.reset()
        if pose_aligner is not None:
            pose_aligner.reset()

        # Step 1: Scan for skier entry frame
        entry_idx, entry_frame, init_box = find_skier_entry_frame(video_file, pipeline.model)

        if init_box is None:
            print(f"  [SKIP] No skier detected in initial scan of {stem_name}")
            continue

        print(f"  -> Skier detected entering at Frame {entry_idx}")

        frame_h, frame_w = entry_frame.shape[:2]

        # Fill pre-entry frames with zeros / NaNs
        video_keypoints_sequence = [np.zeros((17, 3), dtype=np.float32) for _ in range(entry_idx)]
        video_keypoints21_sequence = [np.zeros((21, 3), dtype=np.float32) for _ in range(entry_idx)]
        empty_quad = np.full((2, 8), np.nan, dtype=np.float32)
        video_ski_quads = [empty_quad.copy() for _ in range(entry_idx)]
        video_ski_conf = [np.zeros(2, dtype=np.float32) for _ in range(entry_idx)]
        video_ski_angle = [np.full(2, np.nan, dtype=np.float32) for _ in range(entry_idx)]
        video_ski_length = [np.zeros(2, dtype=np.float32) for _ in range(entry_idx)]
        video_bboxes = [np.full(4, np.nan, dtype=np.float32) for _ in range(entry_idx)]

        # Step 2: Initialize tracker at entry point
        pipeline.init_tracker(entry_frame, init_box)

        # Open video and fast-forward to entry frame
        cap = cv2.VideoCapture(str(video_file))
        cap.set(cv2.CAP_PROP_POS_FRAMES, entry_idx)

        # Step 3: Process video frames
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Body-only B1 mode: full-frame pose, no tracker/ROI drift.
            frame_kpts = np.zeros((17, 3), dtype=np.float32)
            ski_result = {"skis": []}
            frame_quad = np.full((2, 8), np.nan, dtype=np.float32)
            frame_ski_conf = np.zeros(2, dtype=np.float32)
            frame_ski_angle = np.full(2, np.nan, dtype=np.float32)
            frame_ski_length = np.zeros(2, dtype=np.float32)

            if USE_DETECTED_ROI_POSE:
                frame_kpts, pose_bbox = pose_keypoints_detected_roi(
                    pipeline.model, frame, prev_bbox=pipeline.bbox,
                    aligner=pose_aligner,
                    conf_thres=POSE_CONF, bbox_pad=BODY_BOX_PAD)
                if frame_kpts[:, 2].max() > 0 and pose_bbox is not None:
                    bbox = pose_bbox
                    pipeline.bbox = bbox
                    tracking_ok = True
                else:
                    bbox = pipeline.bbox
                    tracking_ok = False
            else:
                bbox, tracking_ok = pipeline.update(frame)
                if not tracking_ok:
                    re_detected = pipeline.recover_target(frame, pipeline.bbox)
                    if re_detected:
                        bbox = pipeline.bbox
                        tracking_ok = True
                if tracking_ok and bbox is not None:
                    frame_kpts = pose_keypoints_in_roi(
                        pipeline.model, frame, bbox,
                        aligner=pose_aligner if ALIGN_POSE_ROI else None)

            # --- classical-CV ski detection on the same tracked ROI ---
            if tracking_ok and bbox is not None and ski_detector is not None:
                have_pose = frame_kpts[:, 2].max() > 0
                ski_result = ski_detector.detect(frame, bbox, frame_kpts if have_pose else None)
                for ski in ski_result["skis"]:
                    slot = 0 if ski["side"] == "left" else 1
                    frame_quad[slot] = ski["quad"].reshape(-1)
                    frame_ski_conf[slot] = ski["conf"]
                    frame_ski_angle[slot] = ski["angle"]
                    frame_ski_length[slot] = ski["length"]

            video_keypoints21_sequence.append(attach_ski_keypoints(frame_kpts, ski_result))
            video_ski_quads.append(frame_quad)
            video_ski_conf.append(frame_ski_conf)
            video_ski_angle.append(frame_ski_angle)
            video_ski_length.append(frame_ski_length)
            video_bboxes.append(np.asarray(bbox if bbox is not None else [np.nan] * 4, dtype=np.float32))
            video_keypoints_sequence.append(frame_kpts)

        cap.release()

        # Step 4: Export arrays
        if len(video_keypoints_sequence) > 0:
            seq_array = np.array(video_keypoints_sequence, dtype=np.float32)  # [T, 17, 3]
            save_file = output_path / f"{stem_name}_pose.npy"
            np.save(save_file, seq_array)

            # 21-keypoint layout expected by compute_skier_orientation_features
            pose21_array = np.array(video_keypoints21_sequence, dtype=np.float32)  # [T, 21, 3]
            np.save(output_path / f"{stem_name}_pose21.npy", pose21_array)

            # ski tracks: 2 skis x 4 quad points, plus confidence / angle / length
            ski_file = SKI_OUTPUT_DIR / f"{stem_name}_ski.npz"
            np.savez_compressed(
                ski_file,
                quads=np.array(video_ski_quads, dtype=np.float32),
                conf=np.array(video_ski_conf, dtype=np.float32),
                angle=np.array(video_ski_angle, dtype=np.float32),
                length=np.array(video_ski_length, dtype=np.float32),
                bbox=np.array(video_bboxes, dtype=np.float32),
                fps=np.float32(cap.get(cv2.CAP_PROP_FPS) or 30.0),
                width=np.int32(frame_w),
                height=np.int32(frame_h),
            )

            processed_records.append({
                "filename": row["filename"],
                "pose_feature_path": str(save_file),
                "ski_feature_path": str(ski_file),
                "num_frames": len(seq_array),
                "target": target_score
            })

    dataset_summary = pd.DataFrame(processed_records)
    summary_path = output_path / "dataset_processed_features.csv"
    dataset_summary.to_csv(summary_path, index=False)
    print(f"\nPipeline finished successfully! Features saved to: {summary_path}")
    return dataset_summary


# Execute complete pipeline ---------------------------------------------------
# Process one source at a time so the broadcast clips use a larger inference
# size.  Existing pose arrays are skipped when OVERWRITE_POSE is False.
dataset_frames = []
for _dataset, _imgsz, _track_mode, _overwrite, _person_select in (
        ("aqa7", POSE_IMGSZ_AQA, TRACK_MODE_AQA, OVERWRITE_POSE_AQA,
         PERSON_SELECT_AQA),
        ("competition", POSE_IMGSZ_COMPETITION, TRACK_MODE_COMPETITION,
         OVERWRITE_POSE_COMPETITION, PERSON_SELECT_COMPETITION)):
    _sub = labels[labels["dataset"] == _dataset]
    if _sub.empty:
        continue
    print(f"\n========== {_dataset}: {len(_sub)} clips "
          f"(imgsz={_imgsz}, track={_track_mode}, person={_person_select}, "
          f"overwrite={_overwrite}) ==========")
    _frames = process_aqa_dataset(_sub, output_dir=OUTPUT_DIR, imgsz=_imgsz,
                                 track_mode=_track_mode, overwrite=_overwrite,
                                 cut_aware=CUT_AWARE,
                                 person_select=_person_select)
    if _frames is not None and len(_frames):
        _frames["dataset"] = _dataset
        dataset_frames.append(_frames)

dataset_features_df = (pd.concat(dataset_frames, ignore_index=True)
                       if dataset_frames else pd.DataFrame())
if len(dataset_features_df):
    dataset_features_df.to_csv(OUTPUT_DIR / "dataset_processed_features.csv",
                               index=False)
    print(f"\nProcessed {len(dataset_features_df)} clips total.")


In [20]:
# ==============================================================================
# VISUAL CHECK: production track + camera shots + air window
# ==============================================================================
# Reads the cached pose arrays / pose_meta.json written by the pipeline cell
# (pass recompute=True to run the tracker again).  Works for AQA and
# competition clips and overlays:
#   * the pose skeleton and the tracked body box,
#   * camera-cut frames; every shot is tracked, nothing is dimmed,
#   * the automatic jump-event air window (green) and the manual annotation
#     window (orange) when available,
#   * a timeline bar with cuts / full tracked span / air windows.
#
# Examples
# --------
# visualize_clip("oly_beijing_2022_men_026")          # by clip id
# visualize_clip("160", dataset="aqa7")               # AQA clip
# visualize_clip(dataset="competition", index=5)      # by position
# visualize_clip("xg_aspen_2026_men_002", recompute=True)
# list_clips(dataset="competition", scored_only=True)

import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Reload so edits to temporal_tracker / ski_detector are picked up without a
# kernel restart (the usual cause of stale-import errors).
import importlib
import temporal_tracker
importlib.reload(temporal_tracker)
from temporal_tracker import (process_video_pose_temporal, find_entry_frame,
                              detect_cuts)
from ski_detector import bbox_from_keypoints

# --- paths / config, reusing the values from the pipeline cell ---------------
POSE_DIR = Path(globals().get("OUTPUT_DIR", "roi_pose_dataset"))
SKI_DIR = Path(globals().get("SKI_OUTPUT_DIR", "ski_dataset"))
DATA_DIR = Path(globals().get("DATA_DIR", "."))
CUT_AWARE_DEFAULT = bool(globals().get("CUT_AWARE", True))
POSE_CONF_DEFAULT = float(globals().get("POSE_CONF", 0.1))
BODY_BOX_PAD_DEFAULT = float(globals().get("BODY_BOX_PAD", 0.25))
MAX_GAP_DEFAULT = int(globals().get("TEMPORAL_MAX_GAP", 15))
POSE_IMGSZ = {"aqa7": int(globals().get("POSE_IMGSZ_AQA", 640)),
              "competition": int(globals().get("POSE_IMGSZ_COMPETITION", 1280))}
TRACK_MODE = {"aqa7": globals().get("TRACK_MODE_AQA", "longest"),
              "competition": globals().get("TRACK_MODE_COMPETITION", "longest")}
PERSON_SELECT = {"aqa7": globals().get("PERSON_SELECT_AQA", "event"),
                 "competition": globals().get("PERSON_SELECT_COMPETITION", "motion")}

if ("labels" not in globals() or not isinstance(labels, pd.DataFrame)
        or labels.empty):
    labels = pd.read_csv(Path("/Users/msserralheiro/Desktop/M_DS/CV/CV_Project/labels_xg2025men.csv"), sep=",")
    labels["video_path"] = labels["filename"].apply(
        lambda s: DATA_DIR / f"{s}.mp4"
    ).apply(lambda p: p if p.is_file() else None)
    labels["dataset"] = "competition"

COCO_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),             # head
    (5, 6),                                     # shoulders
    (5, 7), (7, 9), (6, 8), (8, 10),            # arms
    (5, 11), (6, 12), (11, 12),                 # torso
    (11, 13), (13, 15), (12, 14), (14, 16),     # legs
]


# --- small drawing helpers ---------------------------------------------------
def _put_text(img, text, org, scale, color, thickness=None):
    t = thickness or max(1, int(round(2 * scale)))
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale,
                (0, 0, 0), t + 2, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale,
                color, t, cv2.LINE_AA)


def _put_text_right(img, text, y, scale, color, margin=None):
    margin = margin or int(round(12 * scale))
    t = max(1, int(round(2 * scale)))
    (tw, _), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, t)
    _put_text(img, text, (img.shape[1] - tw - margin, y), scale, color, t)


def _draw_pose(frame, keypoints, threshold=0.3, scale=1.0):
    if keypoints is None or keypoints.shape[0] < 17:
        return frame
    thick = max(1, int(round(2 * scale)))
    rad = max(2, int(round(4 * scale)))
    for a, b in COCO_SKELETON:
        if keypoints[a, 2] >= threshold and keypoints[b, 2] >= threshold:
            cv2.line(frame, (int(keypoints[a, 0]), int(keypoints[a, 1])),
                     (int(keypoints[b, 0]), int(keypoints[b, 1])),
                     (0, 255, 255), thick, cv2.LINE_AA)
    for x, y, c in keypoints:
        if c >= threshold:
            cv2.circle(frame, (int(x), int(y)), rad, (0, 0, 255), -1)
            cv2.circle(frame, (int(x), int(y)), max(1, rad // 2),
                       (255, 255, 255), -1)
    return frame


# --- air windows -------------------------------------------------------------
_AIR_CACHE = None


def _air_windows():
    """Return (event_windows, manual_windows) keyed by clip filename."""
    global _AIR_CACHE
    if _AIR_CACHE is not None:
        return _AIR_CACHE
    event, manual = {}, {}
    p = Path("jump_features_cal_leakfree.csv")
    if p.is_file():
        j = pd.read_csv(p)
        j["filename"] = j["filename"].astype(str)
        for r in j.itertuples(index=False):
            if pd.notna(getattr(r, "to_rel", np.nan)) and pd.notna(
                    getattr(r, "ld_rel", np.nan)):
                event[str(r.filename)] = (float(r.to_rel), float(r.ld_rel))
    p = Path("jump_annotation_filled.csv")
    if p.is_file():
        a = pd.read_csv(p)
        a = a[a["annotated"] == 1]
        for r in a.itertuples(index=False):
            to = getattr(r, "ann_takeoff_s", np.nan)
            ld = getattr(r, "ann_landing_s", np.nan)
            if pd.notna(to) and pd.notna(ld):
                manual[str(r.filename)] = (float(to), float(ld))
    _AIR_CACHE = (event, manual)
    return _AIR_CACHE


def _window_frames(win, n, fps, kind):
    if win is None:
        return None
    if kind == "rel":
        return (int(round(win[0] * (n - 1))), int(round(win[1] * (n - 1))))
    return (int(round(win[0] * fps)), int(round(win[1] * fps)))


# --- clip selection / tracking ----------------------------------------------
_VIZ_MODELS = {}


def _get_model(imgsz):
    imgsz = int(imgsz)
    if imgsz not in _VIZ_MODELS:
        m = YOLO("yolov8n-pose.pt")
        m.overrides["imgsz"] = imgsz
        _VIZ_MODELS[imgsz] = m
    return _VIZ_MODELS[imgsz]


def list_clips(dataset=None, scored_only=False, has_pose=False, limit=20):
    """List selectable clips (handy to find an index for visualize_clip)."""
    df = labels[labels["video_path"].notna()]
    if dataset:
        df = df[df["dataset"] == dataset]
    if scored_only:
        df = df[df["target"].notna()]
    if has_pose:
        df = df[df["video_path"].map(
            lambda p: (POSE_DIR / f"{Path(str(p)).stem}_pose.npy").is_file()
            if pd.notna(p) else False)]
    cols = [c for c in ("filename", "dataset", "target", "fall", "video_path")
            if c in df.columns]
    return df[cols].head(int(limit)).reset_index(drop=True)


def _select_clip(clip=None, dataset=None, index=0):
    df = labels[labels["video_path"].notna()]
    if dataset:
        df = df[df["dataset"] == dataset]
    if df.empty:
        raise ValueError(f"no clips for dataset={dataset!r}")
    if clip is not None:
        key = str(clip)
        exact = df[df["filename"].astype(str) == key]
        if exact.empty:
            exact = df[df["video_path"].map(
                lambda p: Path(str(p)).stem == key if pd.notna(p) else False)]
        if exact.empty:
            exact = df[df["filename"].astype(str).str.contains(
                key, case=False, regex=False)]
        if exact.empty:
            raise ValueError(f"no clip matching {key!r} (dataset={dataset!r})")
        if len(exact) > 1:
            print("[WARN] several matches, using the first:")
            print(exact[["filename", "dataset"]].head(10).to_string(index=False))
        return exact.iloc[0]
    return df.reset_index(drop=True).iloc[int(index)]


def _track_clip(row, recompute=False, write_cache=False):
    video = Path(str(row["video_path"]))
    stem = video.stem
    pose_path = POSE_DIR / f"{stem}_pose.npy"
    meta_path = POSE_DIR / f"{stem}_pose_meta.json"

    if pose_path.is_file() and not recompute:
        pose = np.load(pose_path)
        meta = (json.loads(meta_path.read_text(encoding="utf-8"))
                if meta_path.is_file() else {})
        return pose, meta, stem, "cache"

    ds = row.get("dataset", "aqa7")
    imgsz = POSE_IMGSZ.get(ds, 640)
    track_mode = TRACK_MODE.get(ds, "longest")
    person_select = PERSON_SELECT.get(ds, "longest")
    print(f"[TRACK] {stem}: imgsz={imgsz}, track={track_mode}, "
          f"person={person_select}, all shots (this can take a few minutes)")
    model = _get_model(imgsz)
    if track_mode == "longest":
        entry_idx, entry_bbox = 0, None
    else:
        entry_idx, entry_bbox = find_entry_frame(video, model)
        if entry_bbox is None:
            raise RuntimeError(f"no skier found in the entry scan of {stem}")
    pose, _boxes, meta = process_video_pose_temporal(
        model, video, entry_idx, entry_bbox,
        max_gap=MAX_GAP_DEFAULT, conf_thres=POSE_CONF_DEFAULT,
        bbox_pad=BODY_BOX_PAD_DEFAULT, track_mode=track_mode,
        cut_aware=CUT_AWARE_DEFAULT, person_select=person_select)
    if write_cache:
        np.save(pose_path, pose)
        meta_path.write_text(json.dumps(meta, indent=1), encoding="utf-8")
    return pose, meta, stem, "computed"


def _load_boxes(stem, pose):
    n = len(pose)
    ski_path = SKI_DIR / f"{stem}_ski.npz"
    if ski_path.is_file():
        try:
            with np.load(ski_path) as d:
                if "bbox" in d and len(d["bbox"]) == n:
                    return d["bbox"].astype(np.float32)
        except Exception:
            pass
    boxes = np.full((n, 4), np.nan, dtype=np.float32)
    for t in range(n):
        b = bbox_from_keypoints(pose[t], pad=BODY_BOX_PAD_DEFAULT)
        if b is not None:
            boxes[t] = b
    return boxes


def _fallback_cuts(video, n):
    """Detect camera cuts on a low-res pass when no pose_meta.json exists."""
    cap = cv2.VideoCapture(str(video))
    small = []
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        small.append(cv2.cvtColor(cv2.resize(frame, (160, 90)),
                                  cv2.COLOR_BGR2GRAY))
    cap.release()
    return [int(c) for c in detect_cuts(small) if 0 <= int(c) < n]


def _selected_span(meta, n):
    """Full tracked span: the pipeline keeps every camera shot, not a window."""
    entry = int(meta.get("entry_frame", 0) or 0)
    return entry, n


# --- timeline ----------------------------------------------------------------
def _draw_timeline(frame, t, n, span, cuts, event, manual, scale):
    h, w = frame.shape[:2]
    x0, x1 = int(20 * scale), w - int(20 * scale)
    y = h - int(18 * scale)
    if x1 <= x0:
        return frame
    cv2.line(frame, (x0, y), (x1, y), (80, 80, 80),
             max(1, int(round(3 * scale))), cv2.LINE_AA)

    def x_of(f):
        return int(round(x0 + (x1 - x0) * (f / max(n - 1, 1))))

    a, b = span
    cv2.line(frame, (x_of(a), y), (x_of(b), y), (255, 128, 0),
             max(2, int(round(6 * scale))), cv2.LINE_AA)
    for c in cuts:
        cv2.line(frame, (x_of(c), y - int(8 * scale)),
                 (x_of(c), y + int(8 * scale)), (255, 255, 255),
                 max(1, int(round(2 * scale))), cv2.LINE_AA)
    if event:
        cv2.line(frame, (x_of(event[0]), y - int(12 * scale)),
                 (x_of(event[1]), y - int(12 * scale)), (0, 255, 0),
                 max(2, int(round(4 * scale))), cv2.LINE_AA)
    if manual:
        cv2.line(frame, (x_of(manual[0]), y - int(20 * scale)),
                 (x_of(manual[1]), y - int(20 * scale)), (0, 165, 255),
                 max(2, int(round(4 * scale))), cv2.LINE_AA)
    cv2.line(frame, (x_of(t), y - int(14 * scale)),
             (x_of(t), y + int(10 * scale)), (0, 0, 255),
             max(1, int(round(2 * scale))), cv2.LINE_AA)
    return frame


# --- main entry point --------------------------------------------------------
def visualize_clip(clip=None, dataset=None, index=0, save_video=True,
                   recompute=False, write_cache=False, show_window=False,
                   show_inset=True, air_window=True,
                   output_dir="ski_visual_checks"):
    """Render one clip's production track, all camera shots and air windows.

    clip       : filename, video stem or a unique substring (None -> use index)
    dataset    : "aqa7" / "competition" (None -> both)
    index      : row position when clip is None
    recompute  : re-run the tracker instead of using the cached pose array
    write_cache: with recompute=True, overwrite the cached pose/meta files
    """
    row = _select_clip(clip=clip, dataset=dataset, index=index)
    video = Path(str(row["video_path"]))
    stem = video.stem
    target = row.get("target", np.nan)
    fall = row.get("fall", np.nan)
    ds = row.get("dataset", "?")

    pose, meta, stem, source = _track_clip(row, recompute=recompute,
                                           write_cache=write_cache)
    n = len(pose)
    boxes = _load_boxes(stem, pose)
    span = _selected_span(meta, n)

    cap = cv2.VideoCapture(str(video))
    fps = float(meta.get("fps") or cap.get(cv2.CAP_PROP_FPS) or 30.0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or meta.get("width") or 1)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or meta.get("height") or 1)
    n_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or n)

    cuts = [int(c) for c in (meta.get("cut_frames") or []) if 0 <= int(c) < n]
    if not cuts and not meta.get("shots"):
        cuts = _fallback_cuts(video, n)

    event_w, manual_w = _air_windows()
    key = str(row["filename"])
    event_frames = _window_frames(event_w.get(key) or event_w.get(stem),
                                  n, fps, "rel") if air_window else None
    manual_frames = _window_frames(manual_w.get(key) or manual_w.get(stem),
                                   n, fps, "abs") if air_window else None

    visible = pose[:, :, 2].max(axis=1) > 0
    a, b = span
    coverage = float(visible[a:b].mean()) if b > a else 0.0

    print(f"\n--- {stem} [{ds}] ---")
    print(f"video   : {video}")
    print(f"score   : {target if pd.notna(target) else 'unscored'}"
          f" | fall: {fall if pd.notna(fall) else '?'}"
          f" | {n} frames @ {fps:.2f} fps ({n_video} in file)")
    print(f"track   : {meta.get('track_mode', TRACK_MODE.get(ds, '?'))}"
          f" | person: {meta.get('person_select', '?')}"
          f" | source: {source}"
          f" | {int(visible.sum())} frames with pose"
          f" | coverage over clip: {100 * coverage:.0f}%")
    gate_checks = int(meta.get("gate_checks", 0) or 0)
    gate_rejects = int(meta.get("gate_rejects", 0) or 0)
    if gate_checks:
        print(f"gate    : {gate_rejects}/{gate_checks} consecutive links "
              f"rejected ({100 * gate_rejects / gate_checks:.1f}%), "
              f"{int(meta.get('gate_splits', 0) or 0)} path splits")
    print(f"cuts    : {cuts if cuts else 'none'}")
    print(f"shots   : {len(meta.get('shots') or [])} tracked"
          f" / {meta.get('n_camera_shots', '?')} camera"
          f" | full clip {a}-{b} ({b - a} frames), all shots kept")
    shots = meta.get("shots") or []
    if shots:
        entry = int(meta.get("entry_frame", 0) or 0)
        print("  start   end  frames  med px  vy/body  centre  trk")
        for s in shots:
            print(f"  {entry + s['start']:5d} {entry + s['end']:5d}"
                  f"  {s.get('frames', 0):6d}"
                  f"  {s.get('median_size', 0.0):6.0f}"
                  f"  {s.get('vy_range_rel', 0.0):7.2f}"
                  f"  {s.get('center_penalty_mean', 0.0):6.3f}"
                  f"  {'*' if s.get('frames') else ''}")
    if event_frames:
        print(f"event air window : {event_frames[0]}-{event_frames[1]} "
              f"({(event_frames[1] - event_frames[0]) / fps:.2f} s)")
    if manual_frames:
        print(f"manual air window: {manual_frames[0]}-{manual_frames[1]} "
              f"({(manual_frames[1] - manual_frames[0]) / fps:.2f} s)")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    writer = None
    win = f"pipeline check - {stem}"
    if show_window:
        try:
            cv2.namedWindow(win, cv2.WINDOW_NORMAL)
        except Exception as exc:
            print(f"[WARN] no display ({exc}); continuing headless")
            show_window = False

    # contact-sheet frames: prefer the manual window, then the event window
    ca, cb = span
    if manual_frames:
        ca, cb = manual_frames
    elif event_frames:
        ca, cb = event_frames
    ca, cb = max(0, ca), min(n - 1, cb)
    if cb <= ca:
        ca, cb = span[0], max(span[0], span[1] - 1)
    contact_idx = sorted({int(round(ca + (cb - ca) * i / 5.0))
                          for i in range(6)})
    contact_idx = [min(max(i, 0), n - 1) for i in contact_idx]
    contact = []

    scale = max(0.5, min(2.0, width / 1280.0))
    inset_size = int(min(200, max(64, width // 5)))
    text_x = int(14 * scale) + (inset_size + int(14 * scale)
                                if show_inset else 0)
    frame_no = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok or frame_no >= n:
            break
        t = frame_no
        display = frame

        box = boxes[t] if t < len(boxes) else np.full(4, np.nan)
        if np.isfinite(box).all():
            x, y, w, h = (int(v) for v in box)
            cv2.rectangle(display, (x, y), (x + w, y + h), (255, 128, 0),
                          max(1, int(round(2 * scale))), cv2.LINE_AA)
            if show_inset:
                m = 0.25
                x1 = max(0, int(x - w * m))
                y1 = max(0, int(y - h * m))
                x2 = min(display.shape[1], int(x + w * (1 + m)))
                y2 = min(display.shape[0], int(y + h * (1 + m)))
                crop = frame[y1:y2, x1:x2]
                if crop.size:
                    inset = cv2.resize(crop, (inset_size, inset_size))
                    display[10:10 + inset_size, 10:10 + inset_size] = inset
                    cv2.rectangle(display, (10, 10),
                                  (10 + inset_size, 10 + inset_size),
                                  (255, 255, 255), 1)
                    _put_text(display, "pose ROI", (12, 10 + inset_size - 6),
                              scale * 0.5, (255, 255, 255), 1)

        if pose[t, :, 2].max() > 0:
            display = _draw_pose(display, pose[t], scale=scale)

        if width < 900:
            # compact layout for the small AQA frames: everything left-aligned
            small = max(0.4, scale * 0.9)
            _put_text(display, f"{ds} | {stem} | f{t}/{n - 1}",
                      (text_x, 16), small, (0, 255, 0))
            score_txt = (f"score {target:.1f}" if pd.notna(target)
                         else "unscored")
            _put_text(display, f"{score_txt} | pose {100 * coverage:.0f}%",
                      (text_x, 32), small, (0, 255, 0))
            _put_text(display,
                      f"{meta.get('track_mode', '?')} | shots {len(shots)}"
                      f" | all shots",
                      (text_x, 48), small, (255, 128, 0))
            status_y, status_y2 = 68, 86
        else:
            t_scale = max(0.5, min(0.8, scale))
            _put_text(display, f"{stem} | f{t}/{n - 1}",
                      (text_x, int(28 * scale)), t_scale, (0, 255, 0))
            score_txt = (f"score {target:.1f}" if pd.notna(target)
                         else "unscored")
            _put_text(display,
                      f"{ds} | {score_txt} | pose {100 * coverage:.0f}%",
                      (text_x, int(54 * scale)), t_scale, (0, 255, 0))
            _put_text_right(display,
                            f"{meta.get('track_mode', '?')}"
                            f" | shots {len(shots)}",
                            int(28 * scale), t_scale, (0, 255, 0))
            _put_text_right(display, "all shots", int(54 * scale),
                            t_scale, (255, 128, 0))
            status_y, status_y2 = int(82 * scale), int(108 * scale)

        if event_frames and event_frames[0] <= t <= event_frames[1]:
            _put_text(display, "EVENT AIR", (text_x, status_y),
                      scale * 0.9, (0, 255, 0))
        if manual_frames and manual_frames[0] <= t <= manual_frames[1]:
            _put_text(display, "MANUAL AIR", (text_x, status_y2),
                      scale * 0.9, (0, 165, 255))

        _draw_timeline(display, t, n, span, cuts, event_frames,
                       manual_frames, scale)

        if save_video:
            if writer is None:
                writer = cv2.VideoWriter(
                    str(out_dir / f"{stem}_pipeline_check.mp4"),
                    cv2.VideoWriter_fourcc(*"mp4v"), fps,
                    (display.shape[1], display.shape[0]))
            writer.write(display)

        if t in contact_idx and len(contact) < 6:
            tw = 480
            th = max(1, int(round(tw * display.shape[0] / display.shape[1])))
            contact.append(cv2.resize(display, (tw, th)))

        if show_window:
            cv2.imshow(win, display)
            keypress = cv2.waitKey(30) & 0xFF
            if keypress in (ord("q"), 27):
                break
            if keypress == ord(" "):
                cv2.waitKey(0)

        frame_no += 1

    cap.release()
    if writer is not None:
        writer.release()
    if show_window:
        cv2.destroyAllWindows()

    sheet_path = None
    if contact:
        while len(contact) < 6:
            contact.append(np.zeros_like(contact[0]))
        rows = [np.hstack(contact[i:i + 2]) for i in range(0, 6, 2)]
        sheet = np.vstack(rows)
        sheet_path = out_dir / f"{stem}_pipeline_contact_sheet.jpg"
        cv2.imwrite(str(sheet_path), sheet)
        print(f"[SAVED] {sheet_path}")
        try:
            from IPython.display import Image, display as ipy_display
            ipy_display(Image(filename=str(sheet_path)))
        except Exception:
            pass

    video_path = out_dir / f"{stem}_pipeline_check.mp4"
    print(f"[DONE] {frame_no} frames rendered"
          + (f"; video: {video_path}" if save_video else ""))
    return {"stem": stem, "dataset": ds, "frames": n, "fps": fps,
            "cuts": cuts, "selected_span": span,
            "event_air": event_frames, "manual_air": manual_frames,
            "coverage": coverage, "video": video_path,
            "contact_sheet": sheet_path}


# Run one example when the cell is executed.  Change clip / index as needed.
EXAMPLE_CLIP = "xg_aspen_2025_men_b001"
EXAMPLE_DATASET = "competition"
visualize_clip(EXAMPLE_CLIP, dataset=EXAMPLE_DATASET, save_video=True,
               show_window=False)
